In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy

Uility Function

In [3]:
def int_to_bits(n, width):
    # LSB -> MSB
    return [(n >> i) & 1 for i in range(width)]

def bits_to_int(bits):
    # bits: [b0, b1, b2, ...] (LSB -> MSB)
    value = 0
    for i, b in enumerate(bits):
        value |= (int(b) << i)
    return value

Generate Dataset

In [5]:
def build_dataset():
    X = []
    Y = []

    for a in range(16):
        for b in range(16):
            x_bits = int_to_bits(a, 4) + int_to_bits(b, 4)   # 8-bit input
            y_bits = int_to_bits(a + b, 5)                   # 5-bit output
            X.append(x_bits)
            Y.append(y_bits)

    X = torch.tensor(X, dtype=torch.float32)
    Y = torch.tensor(Y, dtype=torch.float32)
    return X, Y

MLP Model

In [6]:
class AddMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(8, 16)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(16, 5)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)   # sigmoid는 loss에서 처리
        return x

Save Parameters in txt

In [ ]:
def dump_params_to_text(model, path="mlp_params.txt"):
    with open(path, "w", encoding="utf-8") as f:
        for name, param in model.named_parameters():
            arr = param.detach().cpu().tolist()

            # shape 구하기
            if isinstance(arr[0], list):   # 2D
                rows = len(arr)
                cols = len(arr[0])
                f.write(f"{name}  shape=({rows}, {cols})\n")
                for i in range(rows):
                    for j in range(cols):
                        f.write(f"{name}[{i}][{j}] = {arr[i][j]:.8f}\n")
            else:                          # 1D
                size = len(arr)
                f.write(f"{name}  shape=({size},)\n")
                for i in range(size):
                    f.write(f"{name}[{i}] = {arr[i]:.8f}\n")

            f.write("\n")

    print(f"Saved parameters to: {path}")

Accuracy Check

In [8]:
@torch.no_grad()
def evaluate(model, X, Y):
    logits = model(X)
    probs = torch.sigmoid(logits)
    pred_bits = (probs >= 0.5).float()

    bit_acc = (pred_bits == Y).float().mean().item()
    sample_acc = (pred_bits == Y).all(dim=1).float().mean().item()

    return bit_acc, sample_acc, pred_bits

Training

In [10]:
def train():
    torch.manual_seed(0)

    X, Y = build_dataset()
    model = AddMLP()

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    epochs = 5000

    for epoch in range(1, epochs + 1):
        model.train()

        logits = model(X)
        loss = criterion(logits, Y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 200 == 0 or epoch == 1:
            bit_acc, sample_acc, _ = evaluate(model, X, Y)
            print(
                f"Epoch {epoch:4d} | "
                f"Loss: {loss.item():.6f} | "
                f"Bit Acc: {bit_acc*100:.2f}% | "
                f"Sample Acc: {sample_acc*100:.2f}%"
            )

        # 모든 샘플의 5비트가 전부 맞으면 조기 종료
        bit_acc, sample_acc, _ = evaluate(model, X, Y)
        if sample_acc == 1.0:
            print(f"\nReached 100% sample accuracy at epoch {epoch}.")
            break

    return model, X, Y

Example Inference

In [11]:
@torch.no_grad()
def test_examples(model):
    tests = [(0, 0), (3, 5), (7, 8), (9, 6), (15, 15)]

    print("\nExample predictions:")
    for a, b in tests:
        x = torch.tensor([int_to_bits(a, 4) + int_to_bits(b, 4)], dtype=torch.float32)
        logits = model(x)
        probs = torch.sigmoid(logits)
        pred_bits = (probs >= 0.5).int().squeeze(0).tolist()
        pred_value = bits_to_int(pred_bits)

        print(
            f"{a:2d} + {b:2d} = pred {pred_value:2d}, "
            f"bits {pred_bits}, gt {a+b:2d}"
        )

Execute

In [15]:
if __name__ == "__main__":
    model, X, Y = train()

    bit_acc, sample_acc, _ = evaluate(model, X, Y)
    print(f"\nFinal Bit Accuracy   : {bit_acc*100:.2f}%")
    print(f"Final Sample Accuracy: {sample_acc*100:.2f}%")

    test_examples(model)

    # 텍스트로 파라미터 저장
    dump_params_to_text(model, "mlp_params.txt")

Epoch    1 | Loss: 0.700633 | Bit Acc: 49.84% | Sample Acc: 1.56%
Epoch  200 | Loss: 0.290578 | Bit Acc: 87.03% | Sample Acc: 51.17%
Epoch  400 | Loss: 0.170336 | Bit Acc: 93.44% | Sample Acc: 74.22%
Epoch  600 | Loss: 0.141056 | Bit Acc: 94.30% | Sample Acc: 75.78%
Epoch  800 | Loss: 0.127093 | Bit Acc: 94.22% | Sample Acc: 75.00%
Epoch 1000 | Loss: 0.113962 | Bit Acc: 94.69% | Sample Acc: 75.78%
Epoch 1200 | Loss: 0.097283 | Bit Acc: 95.00% | Sample Acc: 78.52%
Epoch 1400 | Loss: 0.090773 | Bit Acc: 95.31% | Sample Acc: 78.52%
Epoch 1600 | Loss: 0.063750 | Bit Acc: 97.97% | Sample Acc: 90.23%
Epoch 1800 | Loss: 0.046619 | Bit Acc: 98.75% | Sample Acc: 93.75%
Epoch 2000 | Loss: 0.041319 | Bit Acc: 98.91% | Sample Acc: 94.53%
Epoch 2200 | Loss: 0.038105 | Bit Acc: 98.83% | Sample Acc: 94.14%
Epoch 2400 | Loss: 0.035144 | Bit Acc: 98.75% | Sample Acc: 93.75%
Epoch 2600 | Loss: 0.033064 | Bit Acc: 98.67% | Sample Acc: 93.36%
Epoch 2800 | Loss: 0.031448 | Bit Acc: 98.75% | Sample Acc: 93.

RuntimeError: Numpy is not available